In [8]:
import pandas as pd
import yfinance as yf
import numpy as np

In [9]:
def ler_df_caminho(caminho):
    df = pd.read_csv(caminho)
    df["Data"] = pd.to_datetime(df["Data"])
    cols_info = ["Mes", "Data", "Modelo"]
    tickers = [c for c in df.columns if c not in cols_info]
    datas = df["Data"].sort_values().tolist()


    inicio = datas[0]
    fim = datas[-1] + pd.Timedelta(days=40)

        # 2. Baixar preços das ações
    precos = yf.download(tickers, start=inicio, end=fim)["Close"]
    
    if isinstance(precos, pd.Series):
        precos = precos.to_frame()
    
    # 4. Expandir pesos ao longo do tempo
    pesos_diarios = pd.DataFrame(0.0, index=precos.index, columns=tickers)
    for i, data_reb in enumerate(datas):
        linha = df[df["Data"] == data_reb]
        if linha.empty:
            raise ValueError(f"Data de rebalanceamento {data_reb.date()} não existe no CSV")
        pesos = linha[tickers].iloc[0].fillna(0)  # Preencher NaN com 0
        
        # Próximo rebalanceamento ou final
        if i < len(datas)-1:
            prox_data = datas[i+1]
        else:
            prox_data = precos.index[-1]
        
        mask = (pesos_diarios.index >= data_reb) & (pesos_diarios.index < prox_data)
        pesos_diarios.loc[mask, :] = pesos.values

    # ✅ CORREÇÃO: Calcular retornos percentuais
    retornos = precos.pct_change()

    # ✅ Calcular retorno diário do portfólio (soma ponderada)
    portfolio_returns = (retornos * pesos_diarios).sum(axis=1)

    # ✅ Calcular valor acumulado começando em 100
    portfolio_close = (1 + portfolio_returns).cumprod() * 100
    portfolio_close.iloc[0] = 100  # Garantir que começa em 100

    return portfolio_close, inicio, fim, tickers

In [10]:
def ler_df_portifolio(caminhos=['portfolios_mensais_RF_2024-10-01_2025-10-01.csv','portfolios_mensais_media3m_2024-10-01_2025-10-01.csv']):
    dados = {}
    tickers = {}
    for caminho in caminhos:
        portfolio_close, inicio, fim, ticker = ler_df_caminho(caminho)
        dados[caminho.split(sep='_')[2]] = portfolio_close
        tickers[caminho.split(sep='_')[2]] = ticker

    
    # 3. Baixar IBOV
    ibov = yf.download("^BVSP", start=inicio, end=fim)["Close"]

    # ✅ Calcular IBOV normalizado
    ibov_returns = ibov.pct_change()
    ibov_close = (1 + ibov_returns).cumprod() * 100
    ibov_close.iloc[0] = 100

    
    dados['IBOV'] = ibov_close['^BVSP']
    # 6. Juntar tudo
    df_final = pd.DataFrame(dados).dropna()

    return df_final,dados,tickers

In [11]:
def compute_drawdown(series):
    cummax = series.cummax()
    dd = (series - cummax) / cummax
    max_dd = dd.min()

    # Encontrar datas do pico e vale
    valley = dd.idxmin()
    peak = series.loc[:valley].idxmax()

    return dd, max_dd, peak, valley


def compute_metrics(series, benchmark=None, rf_rate=0):
    ret = series.pct_change().dropna()

    ann_factor = 252

    # Retornos
    cumulative = series.iloc[-1] / series.iloc[0] - 1
    daily_mean = ret.mean()
    annual_return = (1 + daily_mean)**ann_factor - 1

    # Risco
    daily_vol = ret.std()
    annual_vol = daily_vol * np.sqrt(ann_factor)

    # Drawdown
    dd_series, max_dd, peak, valley = compute_drawdown(series)

    # VaR e ES
    var_95 = ret.quantile(0.05)
    var_99 = ret.quantile(0.01)
    es_95 = ret[ret <= var_95].mean()
    es_99 = ret[ret <= var_99].mean()

    # Sharpe e Sortino
    downside = ret[ret < 0].std()
    sharpe = (daily_mean - rf_rate/252) / daily_vol if daily_vol != 0 else np.nan
    sortino = (daily_mean - rf_rate/252) / downside if downside != 0 else np.nan
    calmar = annual_return / abs(max_dd) if max_dd != 0 else np.nan

    # Informação vs benchmark
    if benchmark is not None:
        bench_ret = benchmark.pct_change().dropna()
        aligned = ret.align(bench_ret, join="inner")
        te = (aligned[0] - aligned[1]).std() * np.sqrt(ann_factor)
        ir = (annual_return - (1 + bench_ret.mean())**252 + 1) / te if te != 0 else np.nan
        corr = aligned[0].corr(aligned[1])
        beta = aligned[0].cov(aligned[1]) / aligned[1].var()
    else:
        te = ir = corr = beta = np.nan
    monthly = series.resample("M").agg(["first", "last"])
    sinal_mensal = (monthly["last"] > monthly["first"]).mean()
    return {
        "Retorno acumulado": cumulative,
        "Retorno anualizado": annual_return,
        "Retorno médio diário": daily_mean,

        "Volatilidade diária": daily_vol,
        "Volatilidade anualizada": annual_vol,

        "Max Drawdown": max_dd,
        "DD pico": peak,
        "DD vale": valley,

        "VaR 95%": var_95,
        "ES 95%": es_95,
        "VaR 99%": var_99,
        "ES 99%": es_99,

        "Sharpe": sharpe,
        "Sortino": sortino,
        "Calmar": calmar,

        "Tracking Error": te,
        "Information Ratio": ir,
        "Correlação com IBOV": corr,
        "Beta vs IBOV": beta,
        "Acerto de sinal mensal": sinal_mensal
    }


def ler_df_metricas(df_series_base_100):
    results = {}
    df = df_series_base_100
    for col in df.columns:
        if col != "IBOV":
            results[col] = compute_metrics(df[col], benchmark=df["IBOV"])
        else:
            results[col] = compute_metrics(df[col])

    # Transformar em DataFrame organizado
    metrics_df = pd.DataFrame(results)

    return metrics_df


In [12]:
def main_visualizacao(caminhos=['portfolios_mensais_RF_2024-10-01_2025-10-01.csv','portfolios_mensais_media3m_2024-10-01_2025-10-01.csv']):
    df_final, dados, tickers = ler_df_portifolio(caminhos)
    df_metricas = ler_df_metricas(df_final)

    return df_final, df_metricas, tickers

In [13]:
df , df_metricas, tickers = main_visualizacao()

C:\Users\gabri\AppData\Local\Temp\ipykernel_5532\859955796.py:13: FutureWarning: YF.download() has changed argument auto_adjust default to True
  precos = yf.download(tickers, start=inicio, end=fim)["Close"]
[*********************100%***********************]  11 of 11 completed
C:\Users\gabri\AppData\Local\Temp\ipykernel_5532\859955796.py:13: FutureWarning: YF.download() has changed argument auto_adjust default to True
  precos = yf.download(tickers, start=inicio, end=fim)["Close"]
[*********************100%***********************]  43 of 43 completed
C:\Users\gabri\AppData\Local\Temp\ipykernel_5532\384148243.py:11: FutureWarning: YF.download() has changed argument auto_adjust default to True
  ibov = yf.download("^BVSP", start=inicio, end=fim)["Close"]
[*********************100%***********************]  1 of 1 completed
C:\Users\gabri\AppData\Local\Temp\ipykernel_5532\3934820450.py:52: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
 

In [14]:
df_metricas

,RF,media3m,IBOV
Retorno acumulado,-0.010326,0.017518,0.061776
Retorno anualizado,0.002254,0.037694,0.073327
Retorno médio diário,0.000009,0.000147,0.000281
Volatilidade diária,0.009973,0.012603,0.009684
Volatilidade anualizada,0.158314,0.200059,0.153727
Max Drawdown,-0.111424,-0.156329,-0.112212
DD pico,2025-05-13 00:00:00,2025-03-19 00:00:00,2024-10-02 00:00:00
DD vale,2025-08-21 00:00:00,2025-05-07 00:00:00,2025-01-03 00:00:00
VaR 95%,-0.014881,-0.019143,-0.014109
ES 95%,-0.019832,-0.028618,-0.020057


In [15]:
tickers

{'RF': ['LREN3.SA',
  'WEGE3.SA',
  'RDOR3.SA',
  'EMBR3.SA',
  'PETR4.SA',
  'ABEV3.SA',
  'SUZB3.SA',
  'RAIL3.SA',
  'ELET3.SA',
  'VALE3.SA',
  'PRIO3.SA'],
 'media3m': ['TRAD3.SA',
  'LREN3.SA',
  'PSSA3.SA',
  'VIVT3.SA',
  'OIBR3.SA',
  'DIRR3.SA',
  'EMBR3.SA',
  'CSNA3.SA',
  'TEND3.SA',
  'SUZB3.SA',
  'PLPL3.SA',
  'TRIS3.SA',
  'PETZ3.SA',
  'VLID3.SA',
  'GGBR4.SA',
  'PETR4.SA',
  'CMIG4.SA',
  'MSFT34.SA',
  'EMAE4.SA',
  'SLCE3.SA',
  'B3SA3.SA',
  'AERI3.SA',
  'TOTS3.SA',
  'CYRE3.SA',
  'BPAC11.SA',
  'BBAS3.SA',
  'EVEN3.SA',
  'ABEV3.SA',
  'ELET3.SA',
  'TIMS3.SA',
  'GFSA3.SA',
  'MGLU3.SA',
  'MOVI3.SA',
  'PCAR3.SA',
  'HBOR3.SA',
  'RENT3.SA',
  'HYPE3.SA',
  'ASAI3.SA',
  'TFCO4.SA',
  'RDOR3.SA',
  'MRVE3.SA',
  'PINE4.SA',
  'SBSP3.SA']}

In [16]:
df

,RF,media3m,IBOV
Date,,,
2024-10-01,100.000000,100.000000,100.000000
2024-10-02,100.600880,101.140810,100.769840
2024-10-03,99.566693,99.473537,99.378844
2024-10-04,99.942772,100.313847,99.469414
2024-10-07,98.788586,99.807150,99.639986
...,...,...,...
2025-10-06,101.365919,106.041001,108.387486
2025-10-07,100.197760,102.159823,106.687800
2025-10-08,100.321287,103.231573,107.283294


In [41]:
import json


TICKERS = [
    "ITUB4.SA", "BBDC4.SA", "BBAS3.SA", "SANB11.SA", "BPAC11.SA", "CXSE3.SA", "BRAP4.SA", "BRSR6.SA", "CRFB3.SA", "PSSA3.SA", "PINE4.SA",
    "PETR4.SA", "PRIO3.SA", "OIBR4.SA", "ELET3.SA", "CMIG4.SA", "CPFE3.SA", "EGIE3.SA", "ENGI11.SA", "GEMA3.SA", "LIGHT3.SA", "TRPL4.SA", "EQTL3.SA",
    "VALE3.SA", "CSNA3.SA", "USIM5.SA", "GGBR4.SA",
    "MGLU3.SA", "LREN3.SA", "ABEV3.SA", "RENT3.SA", "MOVI3.SA", "VVAR3.SA", "PCAR3.SA", "TRIS3.SA",
    "WEGE3.SA", "JBSS3.SA", "MSFT34.SA", "HYPE3.SA", "SLCE3.SA", "PETZ3.SA", "ARZZ3.SA", "TFCO4.SA", "BRML3.SA",
    "RAIL3.SA", "CCRO3.SA", "LOGB3.SA", "ARZZ3.SA", "EMAE4.SA", "ATUS3.SA",
    "MRVE3.SA", "TEND3.SA", "PLPL3.SA", "GFSA3.SA", "TRAD3.SA",
    "VLID3.SA", "BRIV3.SA", "CYRE3.SA", "EVEN3.SA", "HBOR3.SA",
    "VIVT3.SA", "TIMS3.SA", "OIBR3.SA",
    "SUZB3.SA", "SBSP3.SA", "KLABIN11.SA", "FIBR3.SA",
    "TOTS3.SA", "BRPR3.SA", "CLSA3.SA",
    "MBLY3.SA", "BRF3.SA", "SEQL3.SA", "ASAI3.SA",
    "TOTS3.SA", "NTCO3.SA", "BRQT3.SA", "DIRR3.SA", "TRPL4.SA",
    "EMBR3.SA", "AZUL4.SA", "GOLL4.SA",
    "PSSA3.SA", "SULB3.SA", "SGUP3.SA",
    "B3SA3.SA", "MOVI3.SA", "RBRR3.SA", "RDOR3.SA",
    "AGRO3.SA", "AERI3.SA", "POSI3.SA"
]

# Exemplo de dicionário de modelos para tickers
modelos_para_tickers = tickers

# Monta lista de linhas para a tabela
tabela = []
for ticker in TICKERS:
    for modelo, tickers_usados in modelos_para_tickers.items():
        tabela.append({
            "modelo": modelo,
            "acao": ticker,
            "presente": 1 if ticker in tickers_usados else 0
        })

# Salva como JSON
with open("static/tabela_acoes.json", "w", encoding="utf-8") as f:
    json.dump(tabela, f, ensure_ascii=False, indent=2)

print("Arquivo static/tabela_acoes.json gerado com sucesso!")

Arquivo static/tabela_acoes.json gerado com sucesso!


In [17]:
import altair as alt

In [ ]:
df_reset = df.reset_index().rename(columns={'index': 'Date'})
df_melt = df_reset.melt('Date', var_name='Modelo', value_name='Retorno')

highlight = alt.selection_multi(fields=['Modelo'], bind='legend')



C:\Users\gabri\AppData\Local\Temp\ipykernel_5532\1814440737.py:4: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use selection_point instead.
  highlight = alt.selection_multi(fields=['Modelo'], bind='legend')
C:\Users\gabri\AppData\Local\Temp\ipykernel_5532\1814440737.py:16: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use add_params instead.
  ).add_selection(


In [33]:
highlight = alt.selection_multi(fields=['Modelo'], bind='legend')

# Paleta Bloomberg/Goldman Sachs
paleta_bloomberg = [
    "#E0B44C",  # Gold
    "#2F4B7C",
    "#003F5C",
    "#A5A5A5",
    "#2F9E8F",
    "#FF7C43",
    "#D45087",
]

# limites do eixo y (opcional)
y_min = df_melt["Retorno"].min() * 1
y_max = df_melt["Retorno"].max() * 1

base = (
    alt.Chart(df_melt)
    .mark_line(
        strokeWidth=3,
        interpolate="basis"  # linhas suaves
    )
    .encode(
        x=alt.X(
            "Date:T",
            title="Data",
            axis=alt.Axis(
                labelAngle=0,
                titleFontSize=18,
                labelFontSize=14,
                grid=True,
                gridOpacity=0.12,
                gridColor="#CCCCCC",
            ),
        ),
        y=alt.Y(
            "Retorno:Q",
            title="Retorno",
            scale=alt.Scale(domain=[y_min, y_max]),
            axis=alt.Axis(
                titleFontSize=18,
                labelFontSize=14,
                grid=True,
                gridOpacity=0.15,
                gridColor="#CCCCCC",
            ),
        ),
        color=alt.Color(
            "Modelo:N",
            scale=alt.Scale(range=paleta_bloomberg),
            legend=alt.Legend(
                orient="top",
                title="Modelos",
                direction="horizontal",
                columns=3,
                titleFontSize=18,
                labelFontSize=16,
                symbolSize=200,
                padding=10,
            ),
        ),
        opacity=alt.condition(highlight, alt.value(1), alt.value(0.1)),
        tooltip=[
            alt.Tooltip("Modelo:N", title="Modelo"),
            alt.Tooltip("Retorno:Q", format=".2f", title="Retorno"),
            alt.Tooltip("Date:T", format="%d/%m/%Y", title="Data"),
        ],
    )
    .add_selection(highlight)
    .properties(
        width=900,
        height=450,
        title=alt.TitleParams(
            text="Retorno dos Modelos — Estilo Bloomberg",
            anchor="middle",
            fontSize=26,
            fontWeight="bold",
            font="Helvetica",
        ),
    )
    .configure_view(
        strokeWidth=0,
        fill="#FAFAFA",  # fundo claro estilo bloomberg
    )
    .configure_title(
        color="#333333",
        fontSize=26,
        font="Helvetica Neue",
        anchor="middle",
    )
    .configure_axis(
        domain=False,
        tickColor="#333333",
        labelColor="#333333",
        titleColor="#333333",
    )
    .configure_legend(
        titleColor="#333333",
        labelColor="#333333",
        symbolType="circle",
        symbolStrokeWidth=3,
    )
    .interactive()
)

base.save("static/grafico_retorno.html")
base.display()


C:\Users\gabri\AppData\Local\Temp\ipykernel_5532\3916884131.py:1: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use selection_point instead.
  highlight = alt.selection_multi(fields=['Modelo'], bind='legend')
C:\Users\gabri\AppData\Local\Temp\ipykernel_5532\3916884131.py:70: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use add_params instead.
  .add_selection(highlight)


alt.Chart(...)

In [36]:
y_min = df_alpha_melt["Alpha"].min() * 1
y_max = df_alpha_melt["Alpha"].max() * 1

# ================================
# FAIXAS DE FUNDO POSITIVO / NEGATIVO
# ================================

# fundo azul para alpha > 0
fundo_positivo = alt.Chart(pd.DataFrame({
    "y1": [0],
    "y2": [y_max],
    "x1": [df_alpha_melt["Date"].min()],
    "x2": [df_alpha_melt["Date"].max()],
})).mark_rect(
    color="#D6EDFF",   # azul muito suave
    opacity=0.45
).encode(
    x="x1:T",
    x2="x2:T",
    y="y1:Q",
    y2="y2:Q"
)

# fundo vermelho para alpha < 0
fundo_negativo = alt.Chart(pd.DataFrame({
    "y1": [y_min],
    "y2": [0],
    "x1": [df_alpha_melt["Date"].min()],
    "x2": [df_alpha_melt["Date"].max()],
})).mark_rect(
    color="#FFE0E0",   # vermelho suave
    opacity=0.45
).encode(
    x="x1:T",
    x2="x2:T",
    y="y1:Q",
    y2="y2:Q"
)

# ================================
# GRÁFICO DE LINHAS PROFISSIONAL
# ================================

chart_alpha_lines = (
    alt.Chart(df_alpha_melt)
    .mark_line(strokeWidth=3, interpolate="basis")
    .encode(
        x=alt.X(
            "Date:T",
            title="Data",
            axis=alt.Axis(
                labelAngle=0,
                labelFontSize=14,
                titleFontSize=18,
                grid=True,
                gridOpacity=0.12,
                gridColor="#CCCCCC",
            )
        ),
        y=alt.Y(
            "Alpha:Q",
            title="Alpha (Excesso sobre IBOV)",
            scale=alt.Scale(domain=[y_min, y_max]),
            axis=alt.Axis(
                labelFontSize=14,
                titleFontSize=18,
                grid=True,
                gridOpacity=0.12,
                gridColor="#CCCCCC",
            )
        ),
        color=alt.Color(
            "Modelo:N",
            scale=alt.Scale(range=paleta_bloomberg),
            legend=alt.Legend(
                orient="top",
                title="Modelos",
                columns=3,
                direction="horizontal",
                labelFontSize=16,
                titleFontSize=18,
                symbolSize=200,
                padding=10,
            ),
        ),
        opacity=alt.condition(highlight, alt.value(1), alt.value(0.1)),
        tooltip=[
            alt.Tooltip("Modelo:N"),
            alt.Tooltip("Alpha:Q", title="Alpha", format=".4f"),
            alt.Tooltip("Date:T", title="Data", format="%d/%m/%Y"),
        ],
    )
    .add_selection(highlight)
)

# ================================
# COMBINAR FUNDO + LINHAS
# ================================
chart_alpha = (
    (fundo_negativo + fundo_positivo + chart_alpha_lines)
    .properties(
        width=900,
        height=450,
        title=alt.TitleParams(
            text="Alpha dos Modelos — Estilo Bloomberg",
            anchor="middle",
            fontSize=26,
            fontWeight="bold",
            font="Helvetica",
            color="#333"
        ),
    )
    .configure_view(
        strokeWidth=0,
        fill="#F8F9FA"
    )
    .configure_axis(
        domain=False,
        tickColor="#333",
        labelColor="#333",
        titleColor="#333"
    )
    .configure_legend(
        titleColor="#333",
        labelColor="#333",
        symbolType="circle",
        symbolStrokeWidth=3
    )
    .interactive()
)

chart_alpha.save("static/grafico_alpha.html")
chart_alpha.display()


C:\Users\gabri\AppData\Local\Temp\ipykernel_5532\2965076205.py:93: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use add_params instead.
  .add_selection(highlight)


alt.LayerChart(...)

In [28]:
chart_alpha.display()

alt.Chart(...)